In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/two-sigma-connect-rental-listing-inquiries/images_sample.zip
/kaggle/input/competitions/two-sigma-connect-rental-listing-inquiries/Kaggle-renthop.torrent
/kaggle/input/competitions/two-sigma-connect-rental-listing-inquiries/sample_submission.csv.zip
/kaggle/input/competitions/two-sigma-connect-rental-listing-inquiries/test.json.zip
/kaggle/input/competitions/two-sigma-connect-rental-listing-inquiries/train.json.zip


In [2]:
#ライブラリの読み込み
import pandas as pd
import numpy as np

# 可視化
import matplotlib.pyplot as plt
import seaborn as sns

# モデル
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# 保存
import pickle

# テキスト処理
from sklearn.feature_extraction.text import TfidfVectorizer

# 便利系
from tqdm import tqdm
import gc

In [3]:
#データの読み込み
train = pd.read_json("/kaggle/input/competitions/two-sigma-connect-rental-listing-inquiries/train.json.zip")
test = pd.read_json("/kaggle/input/competitions/two-sigma-connect-rental-listing-inquiries/test.json.zip")

In [4]:
#データの確認
print(train.shape)
display(train.head())
print(test.shape)
display(test.head())

(49352, 15)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


(74659, 14)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address
0,1.0,1,79780be1514f645d7e6be99a3de696c5,2016-06-11 05:29:41,Large with awesome terrace--accessible via bed...,Suffolk Street,"[Elevator, Laundry in Building, Laundry in Uni...",40.7185,7142618,-73.9865,b1b1852c416d78d7765d746cb1b8921f,[https://photos.renthop.com/2/7142618_1c45a2c8...,2950,99 Suffolk Street
1,1.0,2,0,2016-06-24 06:36:34,Prime Soho - between Bleecker and Houston - Ne...,Thompson Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7278,7210040,-74.0000,d0b5648017832b2427eeb9956d966a14,[https://photos.renthop.com/2/7210040_d824cc71...,2850,176 Thompson Street
2,1.0,0,0,2016-06-17 01:23:39,Spacious studio in Prime Location. Cleanbuildi...,Sullivan Street,"[Pre-War, Dogs Allowed, Cats Allowed]",40.7260,7174566,-74.0026,e6472c7237327dd3903b3d6f6a94515a,[https://photos.renthop.com/2/7174566_ba3a35c5...,2295,115 Sullivan Street
3,1.0,2,f9c826104b91d868e69bd25746448c0c,2016-06-21 05:06:02,For immediate access call Bryan.<br /><br />Bo...,Jones Street,"[Hardwood Floors, Dogs Allowed, Cats Allowed]",40.7321,7191391,-74.0028,41735645e0f8f13993c42894023f8e58,[https://photos.renthop.com/2/7191391_8c2f2d49...,2900,23 Jones Street
5,1.0,1,81062936e12ee5fa6cd2b965698e17d5,2016-06-16 07:24:27,Beautiful TRUE 1 bedroom in a luxury building ...,Exchange Place,"[Roof Deck, Doorman, Elevator, Fitness Center,...",40.7054,7171695,-74.0095,a742cf7dd3b2627d83417bc3a1b3ec96,[https://photos.renthop.com/2/7171695_089ffee2...,3254,20 Exchange Place


In [5]:
#最低限の前処理（特徴量追加なし）
df = train.copy()

# created を datetime に変換
df["created"] = pd.to_datetime(df["created"])

# created から最低限の情報だけ抽出
df["created_year"] = df["created"].dt.year
df["created_month"] = df["created"].dt.month
df["created_day"] = df["created"].dt.day
df["created_hour"] = df["created"].dt.hour

test_df = test.copy()

test_df["created"] = pd.to_datetime(test_df["created"])
test_df["created_year"] = test_df["created"].dt.year
test_df["created_month"] = test_df["created"].dt.month
test_df["created_day"] = test_df["created"].dt.day
test_df["created_hour"] = test_df["created"].dt.hour

In [6]:
#dtype を変換する
df["building_id"] = df["building_id"].astype("category")
df["manager_id"] = df["manager_id"].astype("category")

test_df["building_id"] = test_df["building_id"].astype("category")
test_df["manager_id"] = test_df["manager_id"].astype("category")

In [7]:
#目的変数を数値に変換
target_map = {"low": 0, "medium": 1, "high": 2}
df["target"] = df["interest_level"].map(target_map)

In [8]:
#使う特徴量（本当に最低限）
features = [
    "bathrooms", "bedrooms", "latitude", "longitude", "price",
    "building_id", "manager_id",
    "created_year", "created_month", "created_day", "created_hour"
]

In [9]:
#created_time の強化
df["created_weekday"] = df["created"].dt.weekday
df["created_weekofyear"] = df["created"].dt.isocalendar().week.astype(int)
df["created_is_weekend"] = (df["created_weekday"] >= 5).astype(int)

test_df["created_weekday"] = test_df["created"].dt.weekday
test_df["created_weekofyear"] = test_df["created"].dt.isocalendar().week.astype(int)
test_df["created_is_weekend"] = (test_df["created_weekday"] >= 5).astype(int)

features += ["created_weekday", "created_weekofyear", "created_is_weekend"]
features = list(dict.fromkeys(features))

In [10]:
#created 系の弱い特徴を削る
cols_to_drop = [
    "created_year",
    "created_is_weekend",
    "created_month",
    "created_weekday"
]

df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

features = [f for f in features if f not in cols_to_drop]

In [11]:
#manager_id の OOF target encoding（3クラス）完全テンプレ
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

def target_encoding_oof(df, test_df, col, target_col="interest_level", n_splits=5, seed=42):
    """
    df: train 由来（target あり、前処理済み）
    test_df: test 由来（target なし、df と同じ前処理済み）
    col: "manager_id" などのカテゴリ列
    target_col: "interest_level"（high/medium/low）
    """

    # 出力用（high, medium, low の3列）
    oof_te = np.zeros((len(df), 3))
    test_te = np.zeros((len(test_df), 3))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for tr_idx, va_idx in kf.split(df):
        tr_x = df.iloc[tr_idx]
        va_x = df.iloc[va_idx]

        # ★ FutureWarning 対策：observed=False を明示
        stats = (
            tr_x.groupby(col, observed=False)[target_col]
            .value_counts(normalize=True)
            .unstack()
            .fillna(0)
        )

        # validation に適用
        oof_te[va_idx] = stats.reindex(va_x[col]).fillna(0).values

    # test 用（train 全体で学習）
    full_stats = (
        df.groupby(col, observed=False)[target_col]
        .value_counts(normalize=True)
        .unstack()
        .fillna(0)
    )
    test_te = full_stats.reindex(test_df[col]).fillna(0).values

    # DataFrame 化
    te_cols = [f"{col}_te_high", f"{col}_te_medium", f"{col}_te_low"]
    df_te = pd.DataFrame(oof_te, columns=te_cols, index=df.index)
    test_te = pd.DataFrame(test_te, columns=te_cols, index=test_df.index)

    return df_te, test_te

In [12]:
#manager_id の OOF target encoding
# --- ① 古い TE 列を削除（重要） ---
for c in ["manager_id_te_high", "manager_id_te_medium", "manager_id_te_low"]:
    if c in df.columns:
        df = df.drop(columns=[c])
    if c in test_df.columns:
        test_df = test_df.drop(columns=[c])

# --- ② OOF target encoding を実行 ---
df_te, test_te = target_encoding_oof(df, test_df, col="manager_id", target_col="interest_level")

# --- ③ df に TE を追加（この時点で重複なし） ---
df = pd.concat([df, df_te], axis=1)
test_df = pd.concat([test_df, test_te], axis=1)

# --- ④ 念のため重複列を削除（保険） ---
df = df.loc[:, ~df.columns.duplicated()]
test_df = test_df.loc[:, ~test_df.columns.duplicated()]

# --- ⑤ features に TE 列を追加（重複は dict で除去） ---
features += ["manager_id_te_high", "manager_id_te_medium", "manager_id_te_low"]
features = list(dict.fromkeys(features))

In [13]:
#building_id OOF target encoding
# --- 古い building_id TE 列を削除 ---
for c in ["building_id_te_high", "building_id_te_medium", "building_id_te_low"]:
    if c in df.columns:
        df = df.drop(columns=[c])
    if c in test_df.columns:
        test_df = test_df.drop(columns=[c])

# --- building_id の OOF target encoding（← これが “定義” に相当） ---
df_te_b, test_te_b = target_encoding_oof(
    df, test_df,
    col="building_id",
    target_col="interest_level"
)

# --- df/test_df に TE を追加 ---
df = pd.concat([df, df_te_b], axis=1)
test_df = pd.concat([test_df, test_te_b], axis=1)

# --- 念のため重複列を削除 ---
df = df.loc[:, ~df.columns.duplicated()]
test_df = test_df.loc[:, ~test_df.columns.duplicated()]

# --- features に building_id TE を追加 ---
features += ["building_id_te_high", "building_id_te_medium", "building_id_te_low"]
features = list(dict.fromkeys(features))

In [14]:
#manager_id_count / building_id_count
# --- count の辞書を作成 ---
manager_count = df["manager_id"].value_counts()
building_count = df["building_id"].value_counts()

# --- df に追加 ---
df["manager_id_count"] = df["manager_id"].map(manager_count)
df["building_id_count"] = df["building_id"].map(building_count)

# --- test_df に追加（未知IDは0） ---
test_df["manager_id_count"] = test_df["manager_id"].map(manager_count).fillna(0)
test_df["building_id_count"] = test_df["building_id"].map(building_count).fillna(0)

# --- features に追加 ---
features += ["manager_id_count", "building_id_count"]
features = list(dict.fromkeys(features))

In [15]:
#manager_id × building_id の交互作用（count）
df["manager_building"] = df["manager_id"].astype(str) + "_" + df["building_id"].astype(str)
test_df["manager_building"] = test_df["manager_id"].astype(str) + "_" + test_df["building_id"].astype(str)

# train + test の結合で count を作る
all_mb = pd.concat([df["manager_building"], test_df["manager_building"]])

mb_count = all_mb.value_counts()

df["manager_building_count"] = df["manager_building"].map(mb_count)
test_df["manager_building_count"] = test_df["manager_building"].map(mb_count)

features += ["manager_building_count"]
features = list(dict.fromkeys(features))

In [16]:
#manager_building の OOF Target Encoding
#1. manager_building を作る
#df["manager_building"] = df["manager_id"].astype(str) + "_" + df["building_id"].astype(str)
#test_df["manager_building"] = test_df["manager_id"].astype(str) + "_" + test_df["building_id"].astype(str)

In [17]:
#2. target_encoding_oof 関数を正しく呼ぶ
df_mb_te, test_mb_te = target_encoding_oof(
    df=df,
    test_df=test_df,
    col="manager_building",
    target_col="interest_level",   # ← ここが正しい
    n_splits=5,
    seed=42
)

In [18]:
#3. df / test_df に結合
df = pd.concat([df, df_mb_te], axis=1)
test_df = pd.concat([test_df, test_mb_te], axis=1)

#重複列を削除
df = df.loc[:, ~df.columns.duplicated()]
test_df = test_df.loc[:, ~test_df.columns.duplicated()]

In [19]:
#4. features に追加
features += [
    "manager_building_te_high",
    "manager_building_te_medium",
    "manager_building_te_low"
]
features = list(dict.fromkeys(features))

In [20]:
#price_log（log1p）を追加
df["price_log"] = np.log1p(df["price"])
test_df["price_log"] = np.log1p(test_df["price"])

features += ["price_log"]
features = list(dict.fromkeys(features))

In [21]:
#price_per_bedroom（価格の正規化）
df["price_per_bedroom"] = df["price"] / (df["bedrooms"] + 1)
test_df["price_per_bedroom"] = test_df["price"] / (test_df["bedrooms"] + 1)

features += ["price_per_bedroom"]
features = list(dict.fromkeys(features))

In [22]:
#photos（写真枚数）
df["num_photos"] = df["photos"].apply(len)
test_df["num_photos"] = test_df["photos"].apply(len)

features += ["num_photos"]
features = list(dict.fromkeys(features))

#features（amenities の数）
df["num_features"] = df["features"].apply(len)
test_df["num_features"] = test_df["features"].apply(len)

features += ["num_features"]
features = list(dict.fromkeys(features))

#description（説明文の長さ）
#文字数
df["description_length"] = df["description"].apply(lambda x: len(str(x)))
test_df["description_length"] = test_df["description"].apply(lambda x: len(str(x)))

#単語数
df["description_word_count"] = df["description"].apply(lambda x: len(str(x).split()))
test_df["description_word_count"] = test_df["description"].apply(lambda x: len(str(x).split()))

features += ["description_length", "description_word_count"]
features = list(dict.fromkeys(features))

In [23]:
#description_length と description_word_count の片方を削る
cols_to_drop = ["description_length"]

df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

features = [f for f in features if f not in cols_to_drop]

In [24]:
# description_length を復元
df["description_length"] = df["description"].apply(lambda x: len(str(x)))
test_df["description_length"] = test_df["description"].apply(lambda x: len(str(x)))

if "description_length" not in features:
    features.append("description_length")

In [25]:
#description 軽量 TF‑IDF（20〜50 次元）テンプレ
#1. TF‑IDF を学習して次元圧縮（SVD）する
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# ① TF-IDF（語彙数を制限して軽量化）
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,   # 語彙数を制限（軽量化）
)

tfidf_train = tfidf.fit_transform(df["description"])
tfidf_test  = tfidf.transform(test_df["description"])

# ② SVD で次元圧縮（20〜50 次元がベスト）今回は30
svd = TruncatedSVD(n_components=30, random_state=42)

svd_train = svd.fit_transform(tfidf_train)
svd_test  = svd.transform(tfidf_test)

In [26]:
#2. 圧縮後の TF‑IDF を df に追加する
# 30 次元の TF-IDF を特徴量として追加
for i in range(30):
    df[f"tfidf_{i}"] = svd_train[:, i]
    test_df[f"tfidf_{i}"] = svd_test[:, i]

In [27]:
#3. features に追加する（重複排除つき）
features += [f"tfidf_{i}" for i in range(30)]
features = list(dict.fromkeys(features))

In [28]:
# 旧SVD 30軸を削除
for i in range(30):
    col = f"tfidf_{i}"
    if col in df.columns:
        df.drop(columns=[col], inplace=True)
        test_df.drop(columns=[col], inplace=True)

In [29]:
# features から古い30軸を削除
for i in range(30):
    col = f"tfidf_{i}"
    if col in features:
        features.remove(col)

In [30]:
#Step 1：アメニティ（features）を文字列に変換
#amenities（features）TF‑IDF → SVD（20軸）
# features（アメニティ）をスペース区切りの文字列に変換
df["amenities_text"] = df["features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
test_df["amenities_text"] = test_df["features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")

In [31]:
#Step 2：TF‑IDF（2000語）
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_amen = TfidfVectorizer(
    max_features=2000,
    stop_words='english'
)

tfidf_train_amen = tfidf_amen.fit_transform(df["amenities_text"])
tfidf_test_amen = tfidf_amen.transform(test_df["amenities_text"])

In [32]:
#Step 3：SVD（20軸）
from sklearn.decomposition import TruncatedSVD

svd_amen = TruncatedSVD(
    n_components=20,
    random_state=42
)

svd_train_amen = svd_amen.fit_transform(tfidf_train_amen)
svd_test_amen = svd_amen.transform(tfidf_test_amen)

In [33]:
#Step 4：df / test_df に追加
for i in range(20):
    df[f"amen_svd_{i}"] = svd_train_amen[:, i]
    test_df[f"amen_svd_{i}"] = svd_test_amen[:, i]

In [34]:
#Step 5：features に追加
features += [f"amen_svd_{i}" for i in range(20)]
features = list(dict.fromkeys(features))  # 重複排除

In [35]:
#display_address / street_address の簡易特徴
# display_address
df["display_len"] = df["display_address"].apply(lambda x: len(str(x)))
test_df["display_len"] = test_df["display_address"].apply(lambda x: len(str(x)))

df["display_words"] = df["display_address"].apply(lambda x: len(str(x).split()))
test_df["display_words"] = test_df["display_address"].apply(lambda x: len(str(x).split()))

df["display_first"] = df["display_address"].apply(lambda x: str(x).split()[0] if len(str(x).split()) > 0 else "")
test_df["display_first"] = test_df["display_address"].apply(lambda x: str(x).split()[0] if len(str(x).split()) > 0 else "")

df["display_last"] = df["display_address"].apply(lambda x: str(x).split()[-1] if len(str(x).split()) > 0 else "")
test_df["display_last"] = test_df["display_address"].apply(lambda x: str(x).split()[-1] if len(str(x).split()) > 0 else "")

# street_address
df["street_len"] = df["street_address"].apply(lambda x: len(str(x)))
test_df["street_len"] = test_df["street_address"].apply(lambda x: len(str(x)))

df["street_words"] = df["street_address"].apply(lambda x: len(str(x).split()))
test_df["street_words"] = test_df["street_address"].apply(lambda x: len(str(x).split()))

df["street_first"] = df["street_address"].apply(lambda x: str(x).split()[0] if len(str(x).split()) > 0 else "")
test_df["street_first"] = test_df["street_address"].apply(lambda x: str(x).split()[0] if len(str(x).split()) > 0 else "")

df["street_last"] = df["street_address"].apply(lambda x: str(x).split()[-1] if len(str(x).split()) > 0 else "")
test_df["street_last"] = test_df["street_address"].apply(lambda x: str(x).split()[-1] if len(str(x).split()) > 0 else "")

# features に追加
addr_cols = [
    "display_len", "display_words", "display_first", "display_last",
    "street_len", "street_words", "street_first", "street_last"
]

for col in addr_cols:
    if col not in features:
        features.append(col)

In [36]:
#display_first / display_last / street_first / street_last を Label Encoding
from sklearn.preprocessing import LabelEncoder

cat_cols = ["display_first", "display_last", "street_first", "street_last"]

for col in cat_cols:
    le = LabelEncoder()
    
    # train + test を結合して fit（未知カテゴリ対策）
    all_values = pd.concat([df[col], test_df[col]]).astype(str)
    le.fit(all_values)
    
    df[col] = le.transform(df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    
    # features に追加
    if col not in features:
        features.append(col)

In [37]:
cols_to_drop = ["display_last", "street_len", "street_words"]

# df / test_df から削除
df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

# features から削除
features = [f for f in features if f not in cols_to_drop]

In [38]:
#データセットの作成
input_x = df[features].reset_index(drop=True)
input_y = df["target"].reset_index(drop=True)
input_id = df[["listing_id"]].reset_index(drop=True)

In [39]:
def train_lgb_classifier(
        input_x,
        input_y,
        input_id,
        params,
        list_nfold=[0,1,2,3,4],
        n_splits=5,
    ):
    
    # OOF（3クラスの確率）
    train_oof = np.zeros((len(input_x), 3))
    metrics = []
    imp = pd.DataFrame()

    # Stratified KFold（分類なので必須）
    cv = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123).split(input_x, input_y))

    for nfold in list_nfold:
        print("-"*20, nfold, "-"*20)

        idx_tr, idx_va = cv[nfold][0], cv[nfold][1]
        x_tr, y_tr, id_tr = input_x.loc[idx_tr, :], input_y[idx_tr], input_id.loc[idx_tr, :]
        x_va, y_va, id_va = input_x.loc[idx_va, :], input_y[idx_va], input_id.loc[idx_va, :]

        print(x_tr.shape, y_tr.shape, id_tr.shape)
        print(x_va.shape, y_va.shape, id_va.shape)

        # モデル（分類）
        model = lgb.LGBMClassifier(**params)

        model.fit(
            x_tr,
            y_tr,
            eval_set=[(x_tr, y_tr), (x_va, y_va)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=True),
                lgb.log_evaluation(100),
            ],
        )

        # モデル保存
        fname_lgb = f"model_lgb_fold{nfold}.pickle"
        with open(fname_lgb, "wb") as f:
            pickle.dump(model, f, protocol=4)

        # 予測（train）
        y_tr_pred = model.predict_proba(x_tr)
        y_va_pred = model.predict_proba(x_va)

        # logloss（分類）
        logloss_tr = log_loss(y_tr, y_tr_pred)
        logloss_va = log_loss(y_va, y_va_pred)

        metrics.append([nfold, logloss_tr, logloss_va])

        print("[logloss] tr:{:.4f}, va:{:.4f}".format(logloss_tr, logloss_va))

        # OOF（確率）
        train_oof[idx_va] = y_va_pred

        # imp（foldごと）
        _imp = pd.DataFrame({
            "col": input_x.columns,
            "imp": model.feature_importances_,
            "nfold": nfold
        })
        imp = pd.concat([imp, _imp], axis=0)

    # --- fold loop 終了後にまとめて計算 ---
    print("-"*20, "result", "-"*20)

    metrics = np.array(metrics)
    print(metrics)

    print("[cv] logloss_tr:{:.4f}+-{:.4f}, logloss_va:{:.4f}+-{:.4f}".format(
        metrics[:,1].mean(), metrics[:,1].std(),
        metrics[:,2].mean(), metrics[:,2].std(),
    ))

    # OOF 評価
    logloss_oof = log_loss(input_y, train_oof)
    print("[oof] logloss:{:.4f}".format(logloss_oof))

    # OOF 出力
    train_oof = pd.concat([
        input_id.reset_index(drop=True),
        pd.DataFrame(train_oof, columns=["pred_low", "pred_medium", "pred_high"])
    ], axis=1)

    # imp 集計
    imp = imp.groupby("col")["imp"].agg(["mean", "std"]).reset_index(drop=False)
    imp.columns = ["col", "imp", "imp_std"]

    return train_oof, imp, metrics

In [40]:
#パラメータを設定
params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
}

In [41]:
#学習の実行
train_oof, imp, metrics = train_lgb_classifier(
    input_x=input_x,
    input_y=input_y,
    input_id=input_id,
    params=params,
)

-------------------- 0 --------------------
(39481, 53) (39481,) (39481, 1)
(9871, 53) (9871,) (9871, 1)
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignor

In [42]:
#説明変数の重要度の確認
imp.sort_values("imp", ascending=False)[:10]

,col,imp,imp_std
41,manager_id,2200.0,15.215124
22,building_id,1640.4,20.391175
50,price_per_bedroom,634.4,9.476286
48,price,625.0,11.157957
36,longitude,260.0,16.537835
43,manager_id_te_high,237.8,6.572671
24,building_id_te_high,216.4,14.397917
44,manager_id_te_low,209.4,17.198837
45,manager_id_te_medium,202.8,8.584870
26,building_id_te_medium,161.2,10.592450
